# 11. Gépi Tanulás és Térbeli Arbitrázs (Random Forest)

**Cél:** nemlineáris lakásár-előrejelző betanítása és az „alulárazott" (arbitrázs) ingatlanok azonosítása.

---

### 📖 A Random Forest előnye a regressziókkal szemben
Az ökonometriai modellek merev algebrai szabályokat követnek; a Random Forest döntési fák százait építi, így képes felismerni a nemlineáris és komplex interakciós szabályokat is.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

df = load_szamitott_master()
display(HTML(render_area_header_html(df=df)))
elado = df[df['listing_type'] == 'elado'].copy()
print(f"Modellezésre elérhető eladó lakások: {len(elado)} db.")

### 1. Modellpontosság és Feature Importance

**📌 Teljesítmény-metrikák:**
A teszt R² a modell magyarázóerejét, a feature importance diagram pedig azt mutatja, mely változók hordozták a legnagyobb prediktív erőt.

**🔍 Térbeli arbitrázs:**
A predikált és a tényleges hirdetési ár összevetése azonosítja a modell által magasabbra értékelt — potenciálisan alulárazott — ingatlanokat (vételi lehetőség).

In [ ]:
df_pontos = elado[elado['minta_garantalt_pontos'] == 1].copy()
df_pontos['emelet_szam'] = df_pontos['emelet_szam'].fillna(df_pontos['emelet_szam'].median())
df_pontos['epulet_kora_ev'] = df_pontos['epulet_kora_ev'].fillna(df_pontos['epulet_kora_ev'].median())
df_pontos['allapot_kod'] = df_pontos['allapot_kod'].fillna(df_pontos['allapot_kod'].median())

# Kánon-features: a fókuszváltozók kikerültek; helyette az immissziós zóna dummy
features = [c for c in [
    'korrigalt_alapterulet_nm', 'szobaszam_osszes', 'is_panel',
    'has_lift', 'has_erkely', 'allapot_kod', 'emelet_szam', 'epulet_kora_ev',
    'tavolsag_metro_halozati_m', 'tavolsag_vasut_m', 'tavolsag_vasut_halozati_m',
    'is_vasut_immisszio_150m'
] if c in df_pontos.columns]
df_ml = df_pontos.dropna(subset=['nm_ar_huf', 'price_huf'] + features).copy()

X = df_ml[features]
y = df_ml['nm_ar_huf']

from sklearn.model_selection import RepeatedKFold, GroupKFold, cross_val_predict
from sklearn.cluster import KMeans

# (A) Ismételt K-szoros CV (nem-térbeli) — az általános előrejelző képességhez
rkf = RepeatedKFold(n_splits=5, n_repeats=1, random_state=42)  # 1 ismétlés: futásidő

# (B) TÉRBELI blokk-CV: a szomszédos pontok azonos blokkba kerülnek,
# így a tanító/teszt halmazok térben függetlenek (szivárgásmentes becslés)
mid_lat = float(df_ml['geokodolt_lat'].median())
blk = KMeans(n_clusters=8, random_state=42, n_init=10).fit_predict(
    np.column_stack((df_ml['geokodolt_lon'] * 111320.0 * np.cos(np.radians(mid_lat)),
                     df_ml['geokodolt_lat'] * 110540.0))
)
gkf = GroupKFold(n_splits=8)

rf = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
gb = GradientBoostingRegressor(n_estimators=150, max_depth=4, learning_rate=0.08, random_state=42)

def cv_metrics(cv, name, g=None):
    # cross_validate: az ismételt CV-vel is működik (a cross_val_predict csak
    # partíciókra használható, az ismételt CV-ben minden minta többször teszt).
    from sklearn.model_selection import cross_validate
    scorers = ('r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error')
    s_rf = cross_validate(rf, X, y, cv=cv, groups=g, scoring=scorers, n_jobs=-1)
    s_gb = cross_validate(gb, X, y, cv=cv, groups=g, scoring=scorers, n_jobs=-1)
    return {
        'CV': name,
        'RF R²': s_rf['test_r2'].mean(), 'RF MAE': -s_rf['test_neg_mean_absolute_error'].mean(),
        'RF RMSE': -s_rf['test_neg_root_mean_squared_error'].mean(),
        'GB R²': s_gb['test_r2'].mean(), 'GB MAE': -s_gb['test_neg_mean_absolute_error'].mean(),
        'GB RMSE': -s_gb['test_neg_root_mean_squared_error'].mean(),
    }

cv_rows = [cv_metrics(rkf, 'Ismételt 5×1 (nem-térbeli)'), cv_metrics(gkf, 'Térbeli blokk 8×', g=blk)]
cv_df = pd.DataFrame(cv_rows)
display(HTML(cv_df.round(3).to_html(index=False)))
print("A térbeli blokk-CV R²-e a szivárgásmentes alsó becslés; ha érdemben alacsonyabb "
      "a nem-térbelinél, a szomszédsági szivárgás felfújta a naiv mérőszámokat.")

# Fő modellek a teljes mintán (a végső predikciókhoz) + OOS CV-predikció az arbitrázshoz
rf.fit(X, y)
gb.fit(X, y)
y_pred_rf_cv = cross_val_predict(rf, X, y, cv=GroupKFold(n_splits=8), groups=blk)
y_pred_gb_cv = cross_val_predict(gb, X, y, cv=GroupKFold(n_splits=8), groups=blk)

r2_rf = r2_score(y, y_pred_rf_cv)
mae_rf = mean_absolute_error(y, y_pred_rf_cv)
mape_rf = np.mean(np.abs((y - y_pred_rf_cv) / y)) * 100
r2_gb = r2_score(y, y_pred_gb_cv)
mae_gb = mean_absolute_error(y, y_pred_gb_cv)

kpi_cards = [
    ("Random Forest R² (OOS, térbeli CV)", f"{r2_rf:.3f}", "Szivárgásmentes becslés", "#1e3a8a"),
    ("Gradient Boosting R² (OOS, térbeli CV)", f"{r2_gb:.3f}", "Szivárgásmentes becslés", "#2563eb"),
    ("Átlagos Hiba (MAE)", fmt_huf(mae_rf), "Random Forest, OOS", "#059669"),
    ("Relatív Hiba (MAPE)", f"{mape_rf:.1f}%", "Random Forest, OOS", "#10b981"),
    ("Modellezési Minta (N)", f"{len(df_ml)} db", "Garantált pontos eladó", "#d97706"),
    ("Feature-ek száma", f"{len(features)} db", "Kánon változókészlet", "#7c3aed")
]
display(HTML(kpi_grid_html(kpi_cards)))


### 2. Változók Relatív Fontossága (Feature Importance MDI)

**📌 Mely tulajdonságok a legfontosabbak?**
1. Korrigált alapterület (m²) – a legfőbb magyarázó erő,
2. Lokáció (metró és vasút távolság),
3. Műszaki állapotindex,
4. Építőanyag (panel vs. tégla).

In [ ]:
# 1. Tényleges vs Becsült ábra
fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=y / 1e3, y=y_pred_rf_cv / 1e3,
    mode='markers', marker=dict(color='#2563eb', opacity=0.7, size=8),
    name='OOS CV-predikció'
))
min_p = min(y.min(), y_pred_rf_cv.min()) / 1e3
max_p = max(y.max(), y_pred_rf_cv.max()) / 1e3
fig1.add_trace(go.Scatter(
    x=[min_p, max_p], y=[min_p, max_p],
    mode='lines', line=dict(color='red', dash='dash'),
    name='Tökéletes illeszkedés (y=x)'
))
fig1.update_layout(
    title='Random Forest: Tényleges vs. Becsült Négyzetméterár (OOS térbeli CV-predikció)',
    xaxis_title='Tényleges Ár/m² (ezer Ft)',
    yaxis_title='Becsült Ár/m² (ezer Ft)',
    template=PLOTLY_TEMPLATE, height=450
)
fig1.show()

# 2. Feature Importance
feat_names_hu = {
    'korrigalt_alapterulet_nm': 'Alapterület (m²)',
    'allapot_kod': 'Műszaki Állapot',
    'is_panel': 'Panel Épület (dummy)',
    'tavolsag_metro_halozati_m': 'Metró Távolság (m)',
    'tavolsag_vasut_m': 'Vasúti pálya légvonal (zaj)',
    'is_vasut_immisszio_150m': 'Közvetlen immisszió (<150m)',
    'has_erkely': 'Erkély (dummy)',
    'emelet_szam': 'Emelet',
    'epulet_kora_ev': 'Épületkor',
    'tavolsag_vasut_halozati_m': 'Vasútállomás Hálózat (TOD)',
    'szobaszam_osszes': 'Szobaszám',
    'has_lift': 'Lift (dummy)'
}
# Permutációs fontosság a TESZT-halmazon (a Gini-importancia torzított — folytonos változók felé)
from sklearn.model_selection import train_test_split as _tts
_Xtr, _Xte, _ytr, _yte = _tts(X, y, test_size=0.20, random_state=42)
_rf_hold = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1).fit(_Xtr, _ytr)
from sklearn.inspection import permutation_importance
_pi = permutation_importance(_rf_hold, _Xte, _yte, n_repeats=10, random_state=42, n_jobs=1)
importances = pd.DataFrame({
    'Valtozo': [feat_names_hu.get(f, f) for f in features],
    'Fontossag': _pi.importances_mean
}).sort_values('Fontossag', ascending=True)

fig2 = px.bar(
    importances, x='Fontossag', y='Valtozo', orientation='h',
    title='Ármeghatározó Tényezők Relatív Fontossága (Random Forest Gini Importance)',
    labels={'Fontossag': 'Relatív Fontosság (0 - 1)', 'Valtozo': 'Jellemző'},
    color='Fontossag', color_continuous_scale='Blues',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 3. A Legjobb Piaci Arbitrázs Lehetőségek Toplistája

**📌 Mit tartalmaz a táblázat?**
A modell által azonosított legnagyobb mértékben alulárazott kőbányai lakások listáját a hirdetett és becsült ár különbségével.

In [ ]:
# Előrejelzés OUT-OF-SAMPLE térbeli CV-predikcióval (nem in-sample illesztés!)
# — az in-sample predikció a modell hibáját „arbitrázsként" mutatná.
df_ml['becsult_nm_ar'] = y_pred_rf_cv
df_ml['becsult_ar_mft'] = (df_ml['becsult_nm_ar'] * df_ml['korrigalt_alapterulet_nm']) / 1e6
df_ml['ar_kulonbseg_mft'] = df_ml['becsult_ar_mft'] - df_ml['ar_millio_ft']
df_ml['alularazottsag_pct'] = (df_ml['ar_kulonbseg_mft'] / df_ml['becsult_ar_mft']) * 100

# Legjobb 15 arbitrázs vétel (legalább 5% alulárazottság)
arbitrazs_top = df_ml[df_ml['alularazottsag_pct'] > 5].sort_values('alularazottsag_pct', ascending=False).head(15).copy()

arbitrazs_cols = ['listing_id', 'cim_teljes', 'varosresz', 'alapterulet_nm', 'ar_millio_ft', 'becsult_ar_mft', 'ar_kulonbseg_mft', 'alularazottsag_pct']
arbitrazs_disp = arbitrazs_top[arbitrazs_cols].copy()
arbitrazs_disp.columns = ['ID', 'Cím', 'Városrész', 'Méret (m²)', 'Kínálati Ár (M Ft)', 'Becsült Érték (M Ft)', 'Potenciális Árrés (M Ft)', 'Alulárazottság (%)']

html_arb = "<div style='overflow-x:auto; margin: 15px 0;'>" + arbitrazs_disp.round(1).to_html(classes='table table-bordered table-striped table-hover', index=False) + "</div>"
display(HTML("<b>Top 15 alulárazott célpont (OOS térbeli CV-predikció, >5% alulárazottság):</b>" + html_arb))
display(HTML("<p style='color:#64748b; font-size:12px;'>Megjegyzés: az alulárazottság out-of-sample predikciókból számolt — a maradék eltérés előrejelzési bizonytalanságot is tartalmazhat (lásd a kvantilis-intervallumot).</p>"))

# Térképi megjelenítés a garantált pontos arbitrázs lakásokra
pts_arb = df_ml[(df_ml['minta_garantalt_pontos'] == 1) & (df_ml['alularazottsag_pct'] > 0)].copy()
if len(pts_arb) > 0:
    fig3 = px.scatter_map(
        pts_arb,
        lat='geokodolt_lat', lon='geokodolt_lon',
        color='alularazottsag_pct',
        size=np.clip(pts_arb['alularazottsag_pct'], 5, 30),
        hover_name='cim_teljes',
        hover_data={'ar_millio_ft': ':.1f', 'becsult_ar_mft': ':.1f', 'alularazottsag_pct': ':.1f'},
        color_continuous_scale='Viridis',
        zoom=12.2,
        center=get_map_center(),
        map_style='carto-positron',
        title='Alulárazott Ingatlanok Térképi Elhelyezkedése (Alulárazottság mértéke szerint)'
    )
    fig3.update_layout(height=480, margin={"r":0,"t":40,"l":0,"b":0})
    fig3.show()

### 4. Interaktív Árbecslő Kalkulátor

**📌 Próbálja ki az árbecslőt!**
Adja meg a lakás méretét, állapotát, szobaszámát és elhelyezkedését, és a Random Forest modell azonnal megbecsüli a piaci értéket!

In [ ]:
# Értékbecslő motor: lineáris szurrogátum a tényleges feature-készleten + kvantilis intervallum
from sklearn.linear_model import Ridge
import json as _json

surr = Ridge(alpha=0.1).fit(df_ml[features], rf.predict(df_ml[features]))
surr_r2 = r2_score(rf.predict(df_ml[features]), surr.predict(df_ml[features]))
surr_w = {feat: float(coef) for feat, coef in zip(features, surr.coef_)}
surr_b = float(surr.intercept_)
print(f"Ridge-szurrogátum illeszkedése a Random Foresthez: R² = {surr_r2:.3f} "
      + ("(megfelelő)" if surr_r2 > 0.9 else "(a szurrogátum közelítő — a kvantilis sávot lásd lent)"))

# F5-7: kvantilis Gradient Boosting — 90%-os predikciós intervallum (5% és 95% kvantilis)
gb_q05 = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.06,
                                   loss='quantile', alpha=0.05, random_state=42).fit(X, y)
gb_q50 = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.06,
                                   loss='quantile', alpha=0.50, random_state=42).fit(X, y)
gb_q95 = GradientBoostingRegressor(n_estimators=100, max_depth=4, learning_rate=0.06,
                                   loss='quantile', alpha=0.95, random_state=42).fit(X, y)

# Referencia-szcenáriók: 90%-os predikciós intervallum
ref_scen = {
    'Panel, 50 m², jó állapot, <150m zaj': {'korrigalt_alapterulet_nm': 50, 'szobaszam_osszes': 2,
        'is_panel': 1, 'has_lift': 1, 'has_erkely': 0, 'allapot_kod': 4, 'emelet_szam': 3,
        'epulet_kora_ev': 45, 'tavolsag_metro_halozati_m': 800, 'tavolsag_vasut_m': 120,
        'tavolsag_vasut_halozati_m': 900, 'is_vasut_immisszio_150m': 1},
    'Tégla, 65 m², felújított, csendes zóna': {'korrigalt_alapterulet_nm': 65, 'szobaszam_osszes': 3,
        'is_panel': 0, 'has_lift': 0, 'has_erkely': 1, 'allapot_kod': 5, 'emelet_szam': 1,
        'epulet_kora_ev': 20, 'tavolsag_metro_halozati_m': 600, 'tavolsag_vasut_m': 1200,
        'tavolsag_vasut_halozati_m': 1400, 'is_vasut_immisszio_150m': 0},
}
scen_rows = []
for name, vals in ref_scen.items():
    row = pd.DataFrame([{f: vals.get(f, 0) for f in features}])
    scen_rows.append({'Szcenárió': name,
                      'Medián becslés (Ft/m²)': int(gb_q50.predict(row)[0]),
                      '90% intervallum (Ft/m²)': f"{int(gb_q05.predict(row)[0]):,} – {int(gb_q95.predict(row)[0]):,}".replace(',', ' ')})
display(HTML(pd.DataFrame(scen_rows).to_html(index=False)))

_coef = _json.dumps(surr_w)
_intercept = surr_b
_median_nm = float(df_ml['nm_ar_huf'].median())

_html_tpl = '''<div id="ml_app" style="background:#ffffff; border:1px solid #cbd5e1; border-radius:12px; padding:22px; box-shadow:0 4px 6px -1px rgba(0,0,0,0.06); margin:18px 0; font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,Helvetica,Arial,sans-serif;">
  <h3 style="margin:0 0 4px 0; color:#1e3a8a;">Gépi tanulásos értékbecslő (RF lineáris szurrogátum)</h3>
  <p style="margin:0 0 14px 0; color:#64748b; font-size:13px;">A csúszkák a Random Forest Ridge-szurrogátumának valós súlyaival számolnak; az intervallumhoz lásd a kvantilis táblát.</p>
  <div style="display:grid; grid-template-columns:repeat(auto-fit,minmax(250px,1fr)); gap:14px;">
    <div style="background:#f8fafc; padding:14px; border-radius:8px;">
      <div style="font-size:12px; font-weight:700; color:#334155; margin-bottom:6px;">Alapterület: <span id="ml_lbl_area">55 m²</span></div>
      <input type="range" id="ml_area" min="30" max="120" value="55" style="width:100%;" oninput="recalcML()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Szobaszám: <span id="ml_lbl_rooms">2</span></div>
      <input type="range" id="ml_rooms" min="1" max="5" value="2" step="0.5" style="width:100%;" oninput="recalcML()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Állapot (1-6): <span id="ml_lbl_cond">4</span></div>
      <input type="range" id="ml_cond" min="1" max="6" value="4" style="width:100%;" oninput="recalcML()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Épületkor (év): <span id="ml_lbl_age">40</span></div>
      <input type="range" id="ml_age" min="0" max="100" value="40" style="width:100%;" oninput="recalcML()">
    </div>
    <div style="background:#f8fafc; padding:14px; border-radius:8px;">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:0 0 6px;">Vágánytól légvonal (m): <span id="ml_lbl_rail">500</span></div>
      <input type="range" id="ml_rail" min="10" max="2000" value="500" style="width:100%;" oninput="recalcML()">
      <div style="font-size:12px; font-weight:700; color:#334155; margin:10px 0 6px;">Állomás hálózati táv (m): <span id="ml_lbl_railnet">800</span></div>
      <input type="range" id="ml_railnet" min="100" max="2500" value="800" style="width:100%;" oninput="recalcML()">
      <label style="display:block; font-size:13px; margin:10px 0 6px;"><input type="checkbox" id="ml_panel" onchange="recalcML()"> Panel</label>
      <label style="display:block; font-size:13px; margin-bottom:6px;"><input type="checkbox" id="ml_lift" checked onchange="recalcML()"> Lift</label>
      <label style="display:block; font-size:13px; margin-bottom:6px;"><input type="checkbox" id="ml_balcony" checked onchange="recalcML()"> Erkély</label>
      <label style="display:block; font-size:13px; margin-bottom:6px;"><input type="checkbox" id="ml_near150" onchange="recalcML()"> Közvetlen immisszió (<150 m)</label>
    </div>
  </div>
  <div style="margin-top:16px; background:#eff6ff; border:1px solid #93c5fd; border-radius:10px; padding:14px 18px;">
    <div style="font-size:11px; font-weight:700; color:#1e40af; text-transform:uppercase;">Becsült fajlagos ár (szurrogátum)</div>
    <div id="ml_result" style="font-size:26px; font-weight:800; color:#1d4ed8;">-- Ft/m²</div>
    <div id="ml_vs" style="font-size:12px; color:#2563eb;">--</div>
  </div>
</div>
<script>
(function() {
  const w = __COEF__;
  const b = __INTERCEPT__;
  const medNM = __MEDIAN_NM__;
  window.recalcML = function() {
    let p = b;
    p += w['korrigalt_alapterulet_nm'] * +document.getElementById('ml_area').value;
    p += w['szobaszam_osszes'] * +document.getElementById('ml_rooms').value;
    p += w['allapot_kod'] * +document.getElementById('ml_cond').value;
    p += w['epulet_kora_ev'] * +document.getElementById('ml_age').value;
    p += w['tavolsag_vasut_m'] * +document.getElementById('ml_rail').value;
    p += w['tavolsag_vasut_halozati_m'] * +document.getElementById('ml_railnet').value;
    p += w['tavolsag_metro_halozati_m'] * 800;
    p += w['emelet_szam'] * 3;
    p += w['is_panel'] * (document.getElementById('ml_panel').checked ? 1 : 0);
    p += w['has_lift'] * (document.getElementById('ml_lift').checked ? 1 : 0);
    p += w['has_erkely'] * (document.getElementById('ml_balcony').checked ? 1 : 0);
    p += w['is_vasut_immisszio_150m'] * (document.getElementById('ml_near150').checked ? 1 : 0);
    document.getElementById('ml_lbl_area').innerText = document.getElementById('ml_area').value + ' m²';
    document.getElementById('ml_lbl_rooms').innerText = document.getElementById('ml_rooms').value;
    document.getElementById('ml_lbl_cond').innerText = document.getElementById('ml_cond').value;
    document.getElementById('ml_lbl_age').innerText = document.getElementById('ml_age').value;
    document.getElementById('ml_lbl_rail').innerText = document.getElementById('ml_rail').value;
    document.getElementById('ml_lbl_railnet').innerText = document.getElementById('ml_railnet').value;
    document.getElementById('ml_result').innerText = Math.round(p).toLocaleString('hu-HU') + ' Ft/m²';
    document.getElementById('ml_vs').innerText = 'A mintamediánhoz képest: ' + ((p / medNM - 1) * 100 >= 0 ? '+' : '') + ((p / medNM - 1) * 100).toFixed(1) + '%';
  };
  setTimeout(recalcML, 50);
})();
</script>'''
html_calc = (_html_tpl.replace('__COEF__', _coef)
                     .replace('__INTERCEPT__', str(_intercept))
                     .replace('__MEDIAN_NM__', str(_median_nm)))
display(HTML(html_calc))
